# 08 — SHAP Explainability & Cross-Model Stability Analysis

This notebook provides interpretation and analysis of feature-level explainability using SHAP values for the top performing regression models: **Ridge, Lasso, Linear Regression, Random Forest, and XGBoost**. 

The implementation mirrors the approach presented in *Kang et al. (2024)* and analyzes features similarly to the way presented in *Janani et al. (2023)*. 

## 1. SHAP Integration into Pipeline

During model evaluation, SHAP analysis is automatically executed for the models listed below:

**Models with SHAP enabled:**
- Ridge Regression
- Lasso Regression
- Linear Regression
- Random Forest Regressor
- XGBoost Regressor

The SHAP computation is triggered inside the `evaluate_and_save()` function. For each model:
if model_name in ["Ridge_regression", "Lasso_regression", "XGBoost_regression", "RandomForest_regression", "Linear_regression"]:
    compute_and_plot_shap(model, X_df, model_name, plot_directory)


SHAP results are saved to:

results/models/[model_name]_shap_summary.csv
results/plots/[model_name]_shap_summary.png
results/plots/[model_name]_shap_bar.png

## 2. SHAP Processing Logic

The `compute_and_plot_shap()` function first converts all features to numeric types and handles missing values using `_ensure_numeric_df()`.

Based on model type:

- **Linear family models (Lasso, Ridge, Linear)** → SHAP uses `shap.Explainer`.
- **Tree-based models (Random Forest, XGBoost)** → Attempts `TreeExplainer`.
    - For XGBoost: kernel fallback is used if tree-based explainer fails.

After SHAP values are computed:

- Mean absolute SHAP values are calculated per feature.
- A ranked CSV is saved.
- Two visualization plots are generated:
  1. Standard SHAP summary plot
  2. Mean |SHAP| bar chart


## 3. Cross-Model Feature Stability

After computing SHAP values for all enabled models, the following code merges the top-10 most influential features per model:

files = glob.glob(str(plot_directory / "*_shap_summary.csv"))
...
top10 = (all_df.groupby("model", group_keys=False)
         .apply(lambda x: x.sort_values("mean_abs_shap", ascending=False).head(10)))

stability = (top10.groupby("feature")["model"].nunique()
             .reset_index(name="n_models_in_top10")
             .sort_values("n_models_in_top10", ascending=False))

The resulting `shap_stability_across_models_5yrs.csv` shows how frequently each feature appears in top-10 SHAP rankings across different algorithms.

This stability measure indicates:

- Which predictors consistently impact well-being regardless of model architecture.
- Which features appear only under specific conditions (e.g., short vs. long-term windows).


## 4. Interpreting SHAP Outputs

High mean_absolute_SHAP → strong influence on prediction.  
Appears in multiple models top-10 → cross-model reliability.  
Features that persist across temporal windows (1yr/5yr/10yr) → long-term policy relevance.

Example interpretation:

- `v014_rawvalue_norm` (Teen births) remained top-ranked across **all models and time windows.**
- `v149_rawvalue_norm` (Disconnected youth) was consistently influential.
- Economic indicators (GDP, disposable income) contributed minimally in most windows.
- The **arts_ratio_norm** appeared in SHAP rankings but typically with low magnitude.

## 5. Summary

- SHAP provides model specific explainability by quantifying each predictor’s impact.
- Cross-model comparison reveals **feature robustness**.
- Teen births and disconnected youth dominate predictive power across all algorithms.
- Short-term (1-year) results remain consistent with 10-year trends, suggesting stable relationships.
- BEA economic variables have weak predictive influence overall, except for the arts engagement ratio in limited scenarios.

In [ ]:

"""
import shap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

def _ensure_numeric_df(X: pd.DataFrame) -> pd.DataFrame:
    # Force everything to float—no strings, no objects.
    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X)
    X = X.copy()
    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    # Drop columns that are entirely NaN (shouldn’t happen, but safe)
    X = X.dropna(axis=1, how="all")
    # If any rows have NaNs, fill with column means (stable for SHAP)
    if X.isna().any().any():
        X = X.fillna(X.mean(numeric_only=True))
    # Ensure dtype float
    return X.astype(float)

def compute_and_plot_shap(model, X, model_name, out_dir, max_display=20):
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    # 1) Ensure numeric & float dtypes
    X = _ensure_numeric_df(X)

    try:
        # 2) Choose explainer by model type
        if hasattr(model, "coef_"):  # linear family
            explainer = shap.Explainer(model, X)
            values = explainer(X)
        elif "xgboost" in model_name.lower():
            try:
                # Try fast TreeExplainer first (preferred)
                explainer = shap.TreeExplainer(model, feature_perturbation="interventional")
                shap_values = explainer.shap_values(X)
                # wrap into a proper Explanation object if needed
                values = shap.Explanation(values=shap_values, data=X, feature_names=X.columns)
            except Exception as inner_e:
                print(f"TreeExplainer failed for XGBoost, fallback to KernelExplainer: {inner_e}")
                # KernelExplainer fallback
                bg = shap.sample(X, 20, random_state=0).values
                f = lambda data: model.predict(pd.DataFrame(data, columns=X.columns))
                explainer = shap.KernelExplainer(f, bg)
                shap_values = explainer.shap_values(X.values, nsamples="auto")
                values = shap.Explanation(values=shap_values, data=X, feature_names=X.columns)
        else:
            # RandomForest / trees
            explainer = shap.TreeExplainer(model, feature_perturbation="interventional")
            values = explainer(X)

        # 3) Save CSV ranking
        mean_abs = np.abs(values.values).mean(axis=0)
        shap_df = pd.DataFrame({"feature": X.columns, "mean_abs_shap": mean_abs})
        shap_df = shap_df.sort_values("mean_abs_shap", ascending=False)
        shap_df.to_csv(Path(out_dir) / f"{model_name}_shap_summary.csv", index=False)

        # 4) Plots
        shap.summary_plot(values, X, max_display=max_display, show=False)
        plt.title(f"SHAP Summary Plot ({model_name})")
        plt.tight_layout()
        plt.savefig(Path(out_dir) / f"{model_name}_shap_summary.png", dpi=150)
        plt.close()

        shap.summary_plot(values, X, plot_type="bar", max_display=max_display, show=False)
        plt.title(f"Mean |SHAP| Values ({model_name})")
        plt.tight_layout()
        plt.savefig(Path(out_dir) / f"{model_name}_shap_bar.png", dpi=150)
        plt.close()

        print(f"SHAP analysis complete for {model_name}")
    except Exception as e:
        print(f"SHAP failed for {model_name}: {e}")
"""